# The noise floor rises with corpus size

The unrelated-bullshit effect, measured: as the corpus grows, the *best-scoring garbage* for a nonsense query gets better-scoring — something is always nearest. At enough scale the garbage band overlaps the real-results band.

Corpus: the 8 STEM problems from `embedding_geometry.ipynb` plus ~96 templated filler sentences across other domains (history, cooking, sports, bio, CS, physics, econ, math).

Maps onto `embeddings_writeup.html` §4 (Finding 2 and the bar chart).

To rerun: `pip install sentence-transformers` (CPU torch is enough).

## Setup

## Build the corpus

## Embed the corpus and two queries

## The effect

Max cosine of the nonsense query over the first N docs, as the corpus grows from 8 to 104. (The plateau at 32 is an artifact of the templated filler repeating domains — in a real corpus the floor keeps climbing.)

## What the nonsense query actually lands on

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [2]:
base = [
    "A ladder 10 ft long leans against a wall. The bottom slides away at 2 ft/s. How fast is the top sliding down?",
    "Find the eigenvalues and eigenvectors of the matrix [[2, 1], [1, 2]] and diagonalize it.",
    "A disease affects 1 in 1000 people. A test is 99% accurate. If you test positive, what is the probability you have the disease?",
    "A ball is thrown at 20 m/s at 45 degrees. Find the range, maximum height, and time of flight.",
    "Evaluate the integral of x^2 e^x dx by integration by parts.",
    "Prove that sqrt(2) is irrational. Extend the argument to sqrt(p) for any prime p.",
    "Given demand Q = 100 - 2P and supply Q = 3P, find the equilibrium price and consumer surplus.",
    "Solve y'' + 4y = 0 with y(0) = 1, y'(0) = 0, and classify the damping regime.",
]

filler_topics = [
    ("history", "Describe the causes of the {x} war and its consequences for {y}."),
    ("cooking", "A recipe for {x} with {y}: timing, temperature, and technique."),
    ("sports",  "Training plan for improving your {x} by focusing on {y} drills."),
    ("bio",     "Explain how {x} regulates {y} in the cell, with one concrete example."),
    ("cs",      "Implement {x} using {y} and analyze its time complexity."),
    ("physics", "Compute the {x} of a system where {y} is varied continuously."),
    ("econ",    "Analyze how a tax on {x} shifts the {y} curve and who bears the burden."),
    ("math",    "Find the {x} of the function involving {y}, justifying each step."),
]
xs = ["supply", "demand", "heat", "momentum", "gradient", "limit", "integral", "derivative",
      "equilibrium", "resonance", "diffusion", "feedback", "oscillation", "conservation"]
ys = ["energy", "price", "velocity", "entropy", "pressure", "current", "population", "signal"]

filler = []
i = 0
for name, tpl in filler_topics:
    for _ in range(12):
        filler.append(tpl.format(x=xs[i % len(xs)], y=ys[(i // 2) % len(ys)]))
        i += 1
corpus = base + filler  # 104 docs
print(f"corpus size: {len(corpus)}")

corpus size: 104


In [3]:
E = model.encode(corpus, normalize_embeddings=True)
q_cookie = model.encode(["best recipe for chocolate chip cookies"], normalize_embeddings=True)[0]
q_ladder = model.encode(["related rates ladder sliding down a wall"], normalize_embeddings=True)[0]

sims_cookie = E @ q_cookie
sims_ladder = E @ q_ladder

In [4]:
for n in [8, 32, 64, 104]:
    print(f"corpus {n:4d} docs: nonsense-query max cosine = {sims_cookie[:n].max():+.3f}")

corpus    8 docs: nonsense-query max cosine = +0.057
corpus   32 docs: nonsense-query max cosine = +0.290
corpus   64 docs: nonsense-query max cosine = +0.290
corpus  104 docs: nonsense-query max cosine = +0.290


In [5]:
print(f"good query top-1 score: {sims_ladder.max():+.3f}   (margin over #2: {sims_ladder.max() - np.sort(sims_ladder)[-2]:+.3f})")
print(f"nonsense query top-3: {np.sort(sims_cookie)[-3:][::-1].round(3)}")
best = int(np.argmax(sims_cookie))
print(f"nonsense query's top hit: \"{corpus[best][:90]}\"")


good query top-1 score: +0.722   (margin over #2: +0.308)
nonsense query top-3: [0.29  0.252 0.199]
nonsense query's top hit: "A recipe for heat with energy: timing, temperature, and technique."
